In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
US Earthquake Catalog — Pre-Analysis (USGS 1985-2025)

Covers: catalog EDA, Gutenberg-Richter law, Omori-Utsu law.
Run as a script  : python US_preanalysis.py
Convert to notebook: python convert_to_notebook.py US_preanalysis.py notebooks/US_preanalysis.ipynb
"""

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns

from src.seismology import fit_gr_law, fit_omori
from src.plotutils import setup_matplotlib, configure_saves, savefig, save_plotly

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

DATA_DIR = Path("data/USGS")
CUT_YEAR = 1985
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
SAVE_PDF: bool = True
SAVE_JPG: bool = True

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "us" / "preanalysis")

## Data Loading and Basic Statistics

In [ ]:
print("Loading US earthquake catalog...")
df = pd.read_csv(DATA_DIR / "us_earthquakes_1985_2025.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df["year"] = df["time"].dt.year
df_net = (
    df[df["year"] >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)

print(f"Loaded {len(df_net):,} earthquakes "
      f"({df_net['year'].min()}–{df_net['year'].max()})")
print(f"RAM: {df_net.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Columns: {list(df_net.columns)}")
display(pd.concat([df_net.head(3), df_net.tail(3)]))

neg = df_net[df_net["depth_km"] < 0]
print(f"\nNegative-depth events: {len(neg)} "
      f"({len(neg)/len(df_net):.2%} — surface-drift artefact in USGS catalog)")

## Temporal Distribution

Annual event counts assess catalog completeness and reveal seismically active
periods driven by major sequences (e.g., Landers 1992, Northridge 1994,
Ridgecrest 2019). Uniform year-on-year rates above $M_c \approx 1.5$ indicate
stable detection capability from 1985 onward.

The depth-per-year box plot is particularly informative for the US: a deepening
median in later years may reflect improved hypocenter resolution, not
a real physical change — a catalog-quality artefact to keep in mind when
interpreting depth assortativity results.

In [ ]:
year_counts = df_net["year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(18, 5))
sns.barplot(x=year_counts.index, y=year_counts.values, color="darkorange", ax=ax)
ax.set_title("Number of Seismic Events (M ≥ 1.5) in the Contiguous US per Year",
             fontsize=16)
ax.set_xlabel("Year")
ax.set_ylabel("Number of Earthquakes")
plt.xticks(rotation=45)
plt.tight_layout()
savefig("temporal_event_counts_us")
plt.show()

fig, ax = plt.subplots(figsize=(18, 5))
sns.boxenplot(data=df_net, x="year", y="magnitude", palette="flare", ax=ax)
ax.set_title("Magnitude Distribution per Year (US)", fontsize=16)
ax.set_xlabel("Year")
ax.set_ylabel("Magnitude")
plt.xticks(rotation=45)
plt.tight_layout()
savefig("temporal_magnitude_dist_us")
plt.show()

fig, ax = plt.subplots(figsize=(18, 5))
sns.boxplot(
    data=df_net, x="year", y="depth_km",
    palette="crest", showfliers=False, ax=ax,
)
ax.set_title("Depth Distribution per Year (US)", fontsize=16)
ax.set_xlabel("Year")
ax.set_ylabel("Depth (km)")
ax.invert_yaxis()
plt.xticks(rotation=45)
plt.tight_layout()
savefig("temporal_depth_dist_us")
plt.show()

## Magnitude and Depth Analysis

The magnitude histogram should follow the Gutenberg-Richter exponential decay
above the completeness threshold $M_c$; the rapid drop below $M_c$ marks
the under-reporting regime that must be excluded from b-value fits.

US depth structure is more heterogeneous than Italy: **crustal** seismicity
(0–30 km) dominates the western US, while **intermediate-depth** events
(30–100 km) appear in the Pacific Northwest (Cascadia subduction) and
deep events (>100 km) in Alaska-influenced zones. The magnitude-vs-depth
scatter and bin-wise box plots quantify how fault regime correlates with size.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(df_net["magnitude"].dropna(), bins=np.arange(1.45, 8.55, 0.1),
             edgecolor="black", color="darkorange", alpha=0.85)
axes[0].set_xlabel("Magnitude")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Magnitude Distribution (US)")

axes[1].hist(df_net["depth_km"].dropna(), bins=100,
             edgecolor="black", color="salmon", alpha=0.85)
axes[1].set_xlabel("Depth (km)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Depth Distribution (US)")
plt.tight_layout()
savefig("magnitude_depth_distributions_us")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(df_net["magnitude"], df_net["depth_km"],
                alpha=0.04, s=2, color="saddlebrown")
axes[0].invert_yaxis()
axes[0].set_xlabel("Magnitude")
axes[0].set_ylabel("Depth (km)")
axes[0].set_title("Magnitude vs Depth (US)")

hb = axes[1].hexbin(df_net["magnitude"], df_net["depth_km"],
                    gridsize=50, cmap="YlOrRd", mincnt=1, bins="log")
plt.colorbar(hb, ax=axes[1], label="log₁₀(count)")
axes[1].invert_yaxis()
axes[1].set_xlabel("Magnitude")
axes[1].set_ylabel("Depth (km)")
axes[1].set_title("Magnitude vs Depth — Density (US)")
plt.tight_layout()
savefig("magnitude_vs_depth_us")
plt.show()

fig, ax = plt.subplots(figsize=(14, 6))
df_net["mag_bin"] = pd.cut(df_net["magnitude"],
                            bins=[1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 8.0])
sns.boxplot(data=df_net, x="mag_bin", y="depth_km", palette="coolwarm",
            showfliers=False, ax=ax)
ax.invert_yaxis()
ax.set_title("Depth Distribution by Magnitude Bin (US)", fontsize=16)
ax.set_xlabel("Magnitude Range")
ax.set_ylabel("Depth (km)")
plt.tight_layout()
savefig("depth_by_magnitude_bin_us")
plt.show()

## Seismicity Map

Interactive spatial overview of events with M ≥ 4 across the contiguous US.
A lower threshold ($M \geq 4$ vs $M \geq 5$ for Italy) is used because the
broader geographic area requires denser sampling to reveal fault-system structure.

Expected spatial pattern: concentrated activity along the San Andreas fault
system (California), Wasatch Front (Utah), New Madrid Seismic Zone (midwest),
and Pacific Northwest — the same regions that should host the highest-degree
nodes in the Abe-Suzuki network.

In [ ]:
mag_cut = 4
df_map  = df_net[df_net["magnitude"] >= mag_cut].copy()
print(f"Mapping {len(df_map):,} earthquakes (M ≥ {mag_cut})...")

fig = px.scatter_map(
    df_map,
    lat="latitude", lon="longitude",
    color="magnitude", size="magnitude",
    color_continuous_scale="matter",
    zoom=3.5, center={"lat": 37.5, "lon": -96.0},
    map_style="carto-positron",
    hover_name="time",
    hover_data={"latitude": ":.3f", "longitude": ":.3f",
                "depth_km": True, "magnitude": True},
    title=f"Interactive Seismic Map: Contiguous US (M ≥ {mag_cut})",
)
fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
save_plotly(fig, "seismicity_map_us")
fig.show()

## Gutenberg-Richter Law

The Gutenberg-Richter relation states $\log_{10} N(\geq M) = a - bM$, where
$N(\geq M)$ is the cumulative count of earthquakes with magnitude $\geq M$,
$a$ (seismicity rate) and $b \approx 1$ globally (Gutenberg & Richter 1944).

**Sensitivity test**: $M_{\max}$ is varied from 4.0 to 7.0 to assess fit
stability. For the US catalog, the expected b-value is close to 1.0 in
tectonic settings (San Andreas transform) but may deviate in volcanic zones
(Yellowstone, Cascades) where fluids lower effective normal stress.
A systematically rising b-value with increasing $M_{\max}$ signals that large
events are over-represented relative to the power-law — possible catalog
heterogeneity across reporting networks.

In [ ]:
max_mags   = [4, 4.5, 5, 5.5, 6, 6.5, 7]
plot_flags = [True, False, False, True, False, False, True]

gr_results = [
    fit_gr_law(df_net, max_mag=m, plot=p)
    for m, p in zip(max_mags, plot_flags)
]
df_gr = pd.DataFrame(gr_results).sort_values("max_mag")
display(df_gr)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(df_gr["max_mag"], df_gr["a_value"], "o-", color="tab:blue", label="a-value")
ax1.set_xlabel("Max Magnitude Used for Fit")
ax1.set_ylabel("a-value", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(df_gr["max_mag"], df_gr["b_value"], "s-", color="tab:red", label="b-value")
ax2.set_ylabel("b-value", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")
plt.title("Gutenberg-Richter Parameters vs Max Magnitude (US)")
ax1.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
savefig("gr_parameters_sensitivity_us")
plt.show()

## Omori Law — Loma Prieta 1989

The Omori-Utsu law describes aftershock rate decay: $n(t) = K / (c + t)^p$,
where $p \approx 1$ globally (Utsu et al. 1995), $K$ is sequence productivity,
and $c$ regularises the rate at $t = 0$.

**Loma Prieta 1989** (M6.9, 18 Oct 1989, Santa Cruz Mountains) is the most
thoroughly studied US aftershock sequence from the pre-digital era.
Its $p$-value serves as a baseline for California crustal seismicity;
$p > 1$ is typical for warm, fluid-rich crust of the San Andreas system.

In [ ]:
# M6.9, Santa Cruz Mountains, California.
# One of the most-studied US aftershock sequences.
print("Fitting Omori law for Loma Prieta 1989...")
res_loma = fit_omori(
    df_net,
    mainshock_time=pd.to_datetime("1989-10-18 00:04:15", utc=True),
    days=60,
    lat_range=(36.0, 38.5),
    lon_range=(-123.5, -120.5),
    event_name="Loma Prieta 1989",
    mag_cut=1.5,
)

## Omori Law — Ridgecrest 2019

**Ridgecrest 2019** (M7.1, 6 Jul 2019, Mojave Desert) produced the most
densely recorded modern US aftershock sequence, with thousands of events
captured by regional broadband and strong-motion networks.
Comparing its fitted $p$-value to Loma Prieta 1989 tests whether aftershock
decay is consistent across different California fault environments
(strike-slip vs oblique-extensional), a prerequisite for treating
the Abe-Suzuki network topology as fault-regime-independent.

In [ ]:
# M7.1, Mojave Desert, California.
# Second-largest earthquake in California since 1999;
# dense, well-recorded aftershock sequence.
print("Fitting Omori law for Ridgecrest 2019...")
res_ridge = fit_omori(
    df_net,
    mainshock_time=pd.to_datetime("2019-07-06 03:19:53", utc=True),
    days=60,
    lat_range=(34.5, 37.0),
    lon_range=(-118.5, -116.0),
    event_name="Ridgecrest 2019",
    mag_cut=1.5,
)

df_omori = pd.DataFrame([
    {"event": "Loma Prieta 1989",  **res_loma},
    {"event": "Ridgecrest 2019",   **res_ridge},
])
print("\nOmori fit comparison:")
display(df_omori)